# Project Deployment: Azure App Service and CI/CD Pipeline
**Project:** EntiLytics  
**Host:** Microsoft Azure (App Service for Containers)  
**Pipeline:** GitHub Actions (CI/CD)  
**Domain:** `https://entilytics.tech`

This notebook serves as the operational manual for the EntiLytics deployment lifecycle. It covers containerization via Docker, automated deployment via GitHub Actions, custom domain configuration, environment variable management, and common operational commands.

## Phase 1: Containerization (Docker)

EntiLytics is deployed as a containerized application. This ensures that the environment behaves identically whether the application is running on a local machine or on Azure.

### 1.1 Dockerfile Configuration

| Configuration | Value | Reason |
|---|---|---|
| Base Image | `python:3.11-slim` | Minimal footprint, avoids bundling CUDA libraries |
| PyTorch Install | CPU-only build via `--index-url` | Azure for Students uses CPU-only VMs; the default PyTorch wheel includes CUDA (~2 GB extra) |
| Port | `8080` | Azure App Service routes external traffic from port `80` to the container's internal port |
| Working Directory | `/app` | All application files are placed here inside the container |
| `PYTHONPATH` | `/app` | Required for Solara to resolve relative imports correctly |

In [ ]:
# File: Dockerfile (project root)
# This cell is for reference only and is not executed.

dockerfile_contents = """
FROM python:3.11-slim

WORKDIR /app

# Install system dependencies required by certain NLP packages
RUN apt-get update && apt-get install -y build-essential && rm -rf /var/lib/apt/lists/*

# Install PyTorch (CPU-only) before other packages to avoid the CUDA build
RUN pip install --no-cache-dir torch==2.x.x --index-url https://download.pytorch.org/whl/cpu

# Copy and install remaining dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the full application
COPY . .

ENV PYTHONPATH=/app

EXPOSE 8080

CMD ["solara", "run", "app.py", "--host", "0.0.0.0", "--port", "8080"]
"""
print(dockerfile_contents)

### 1.2 Key Entries in requirements.txt

```
solara
sqlalchemy
psycopg2-binary       # PostgreSQL adapter; the binary build requires no system compiler
python-dotenv
flair
sentence-transformers
# torch is installed separately in the Dockerfile as a CPU-only build
```

> **Note on torch:** `torch` is intentionally excluded from `requirements.txt` and installed directly in the Dockerfile. This prevents pip from downloading the default CUDA build, which would add approximately 1 GB to the final image size unnecessarily, since Azure for Students does not provide GPU access.

## Phase 2: Container Registry (Azure Container Registry)

Before the application can be deployed to Azure App Service, the Docker image must be stored in a private registry that Azure can access. This project uses **Azure Container Registry (ACR)**.

| Resource | Value |
|---|---|
| Registry Name | `entilyticsreg` |
| Login Server | `entilyticsreg.azurecr.io` |
| Resource Group | `entilytics-rg` |
| Image Name | `entilytics-app` |
| Image Tag | Git commit SHA (e.g., `a1b2c3d4...`) |

In [ ]:
# Commands used to configure the Azure Container Registry.
# These are run once during initial setup.

# Create the registry
# az acr create \
#     --resource-group entilytics-rg \
#     --name entilyticsreg \
#     --sku Basic

# Enable admin access so GitHub Actions can authenticate
# az acr update --name entilyticsreg --admin-enabled true

# Retrieve credentials to store as GitHub Secrets
# az acr credential show --name entilyticsreg

## Phase 3: CI/CD Pipeline (GitHub Actions)

GitHub Actions automates the full build and deployment sequence. Every push to the `main` branch triggers the pipeline.

### 3.1 Pipeline Overview

| Step | Description |
|---|---|
| Trigger | Push to the `main` branch |
| Build | Docker image is built and tagged with the current commit SHA |
| Push | Image is pushed to Azure Container Registry |
| Deploy | Azure App Service is instructed to pull and run the new image |

### 3.2 Required GitHub Secrets

Add these under **Repository Settings > Secrets and variables > Actions**.

| Secret Name | Description |
|---|---|
| `ACR_NAME` | Registry name (e.g., `entilyticsreg`) |
| `ACR_USERNAME` | Admin username from `az acr credential show` |
| `ACR_PASSWORD` | Admin password from `az acr credential show` |
| `AZURE_WEBAPP_PUBLISH_PROFILE` | Publish profile downloaded from the Azure Portal |

### 3.3 Docker Layer Caching

On the first build, Docker downloads and installs all packages from scratch. This takes several minutes due to large NLP libraries. On subsequent builds, Docker reuses cached layers for anything that has not changed in the Dockerfile. Only modified source files cause a rebuild, so routine code changes deploy much faster after the first run.

In [ ]:
# File: .github/workflows/main.yml
# Place this file in the repository to activate the CI/CD pipeline.

workflow_yaml = """
name: Build and Deploy EntiLytics

on:
  push:
    branches:
      - main

permissions:
  id-token: write
  contents: read

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Log in to Azure Container Registry
        uses: docker/login-action@v2
        with:
          registry: ${{ secrets.ACR_NAME }}.azurecr.io
          username: ${{ secrets.ACR_USERNAME }}
          password: ${{ secrets.ACR_PASSWORD }}

      - name: Build and push Docker image
        uses: docker/build-push-action@v3
        with:
          context: .
          push: true
          tags: ${{ secrets.ACR_NAME }}.azurecr.io/entilytics-app:${{ github.sha }}

  deploy:
    runs-on: ubuntu-latest
    needs: build
    steps:
      - name: Deploy to Azure App Service
        uses: azure/webapps-deploy@v2
        with:
          app-name: entilytics
          publish-profile: ${{ secrets.AZURE_WEBAPP_PUBLISH_PROFILE }}
          images: ${{ secrets.ACR_NAME }}.azurecr.io/entilytics-app:${{ github.sha }}
"""
print(workflow_yaml)

## Phase 4: Custom Domain and SSL

The application is served at `https://entilytics.tech` with a managed TLS certificate provided by Azure.

### 4.1 DNS Configuration

The following DNS records must be set with the domain registrar.

| Record Type | Host | Value | Purpose |
|---|---|---|---|
| `A` | `@` | `<Azure Load Balancer IP>` | Points the root domain to Azure |
| `TXT` | `asuid` | `<Azure Domain Verification Code>` | Proves domain ownership to Azure |
| `CNAME` | `www` | `entilytics.azurewebsites.net` | Points the www subdomain to Azure |

> The Azure Load Balancer IP and domain verification code are found in the **Custom Domains** section of the Azure App Service in the Azure Portal.

### 4.2 SSL Certificate

| Setting | Value |
|---|---|
| Certificate Type | Azure Managed Certificate (free) |
| TLS Binding | SNI SSL |
| HTTPS Enforcement | Enabled (HTTP redirects to HTTPS automatically) |

Azure provisions and renews the managed certificate automatically. No manual certificate management is required.

## Phase 5: Environment Variables and Secrets

All sensitive credentials are injected through **Azure App Service Application Settings** rather than being stored in source code or the Docker image. These settings are available inside the container as standard environment variables at runtime.

### 5.1 Required Application Settings

| Variable | Description |
|---|---|
| `DB_HOST` | Hostname of the Azure PostgreSQL Flexible Server |
| `DB_NAME` | Name of the target database |
| `DB_USER` | PostgreSQL administrator username |
| `DB_PASS` | PostgreSQL administrator password |
| `GOOGLE_CLIENT_ID` | OAuth 2.0 Client ID from Google Cloud Console |
| `GOOGLE_CLIENT_SECRET` | OAuth 2.0 Client Secret from Google Cloud Console |
| `REDIRECT_URI` | Must match `https://entilytics.tech/` exactly as registered in Google |
| `WEBSITES_PORT` | Set to `8080` to match the container's exposed port |
| `PYTHONPATH` | Set to `/app` |
| `WEBSITES_CONTAINER_START_TIME_LIMIT` | Set to `1800` (seconds) to allow NLP models sufficient time to load |

In [ ]:
# Set application settings via Azure CLI.
# Replace placeholder values with actual credentials before running.

# az webapp config appsettings set \
#     --name entilytics \
#     --resource-group entilytics-rg \
#     --settings \
#         DB_HOST="<your-db-hostname>" \
#         DB_NAME="<your-db-name>" \
#         DB_USER="<your-db-username>" \
#         DB_PASS="<your-db-password>" \
#         GOOGLE_CLIENT_ID="<your-client-id>" \
#         GOOGLE_CLIENT_SECRET="<your-client-secret>" \
#         REDIRECT_URI="https://entilytics.tech/" \
#         WEBSITES_PORT="8080" \
#         PYTHONPATH="/app" \
#         WEBSITES_CONTAINER_START_TIME_LIMIT="1800"

## Phase 6: Operational Commands

The following Azure CLI commands are used for routine monitoring and maintenance.

In [ ]:
# Check the current running state of the web app
# az webapp show \
#     --name entilytics \
#     --resource-group entilytics-rg \
#     --query state

# Stream live container logs (most useful during startup debugging)
# az webapp log tail \
#     --name entilytics \
#     --resource-group entilytics-rg

# Restart the web app
# az webapp restart \
#     --name entilytics \
#     --resource-group entilytics-rg

# Enable container logging to filesystem
# (required to see Python stdout and tracebacks in log tail)
# az webapp log config \
#     --name entilytics \
#     --resource-group entilytics-rg \
#     --docker-container-logging filesystem \
#     --level verbose

# List all image tags stored in the container registry
# az acr repository show-tags \
#     --name entilyticsreg \
#     --repository entilytics-app

## Phase 7: Deployment Troubleshooting Reference

This section documents issues encountered during deployment and their confirmed resolutions.

### 7.1 Common Errors and Resolutions

| Symptom | Root Cause | Resolution |
|---|---|---|
| `Container did not start within expected time limit` | Startup timeout too short for NLP model loading | Set `WEBSITES_CONTAINER_START_TIME_LIMIT` to `1800` |
| `Logging is not enabled for this container` | App Service container logging was not configured | Run `az webapp log config` with `--docker-container-logging filesystem` |
| Image pull fails with authentication error | App Service was not configured with ACR credentials | Run `az webapp config container set` with the registry URL, username, and password |
| Application error immediately after deploy | Azure is still pulling and extracting the new image | Wait 5 to 15 minutes after GitHub Actions completes before accessing the site |

### 7.2 Deployment Timing Reference

| Stage | Approximate Duration |
|---|---|
| GitHub Actions build (first run) | 15 to 30 minutes |
| GitHub Actions build (subsequent runs, cached) | 2 to 5 minutes |
| Azure image pull and container start | 5 to 15 minutes |
| Total (first deployment) | 30 to 45 minutes |
| Total (subsequent deployments) | 7 to 20 minutes |

> The application is not available at the domain until Azure completes the image pull and the container has fully started. Accessing the URL before this completes will return a `503` or `Application Error` response.